# Shot-by-Shot Video Analysis

Local LLM + Faster-Whisper pipeline for video analysis on Kaggle.

## Requirements
- GPU: T4 x2 (enable in Runtime → Change runtime type → GPU T4 x2)
- Internet: Enable for first-time model download

## Workflow
1. **T4x2 OFF**: Run Cell 1 (install), then Runtime → Restart Session, then Cells 2-3 + Cell A
2. **Turn T4x2 ON** — kernel restarts automatically
3. **Change MODEL_ID** in CONFIGURATION to `/kaggle/input/yoofun/qwen3-vl-30b/`
4. Run from Cell 2 onward (weights load from Kaggle Dataset at disk speed)

## Configuration
Edit Cell 3 to set your preferences.

## Output Files
- `shot_by_shot_output.csv` - Main merged output
- `stage1_descriptions.csv` - Detailed VLM descriptions
- `stage2_audio_descriptions.csv` - Concise AD sentences

In [ ]:
# Install dependencies
# Run this once before anything else.
# Then do: Runtime → Restart Session, and start from Cell 2.
!pip install -q "numpy==1.26.4" --force-reinstall --no-deps
!pip install -U "transformers>=4.51.0" "accelerate>=1.0" "huggingface_hub>=1.22"
!pip install -U "bitsandbytes>=0.46.1" --no-cache-dir
!pip install "unsloth" "unsloth_zoo"
!pip install "scenedetect" "opencv-python-headless>=4.10"
print("\nDONE. Restart session via Runtime menu, then run from Cell 2.")

In [ ]:
# Import libraries
import os
# Retrieve HF_TOKEN from Kaggle Secrets for faster authenticated downloads
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

import json
import torch
import pandas as pd
import numpy as np
import cv2
import base64
from PIL import Image
from scenedetect import detect, AdaptiveDetector

# Check GPU
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.device_count()} device(s)")
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: No GPU detected. Enable GPU T4 x2 in Runtime settings.")

In [ ]:
# CONFIGURATION - Edit these settings
# ============================================

# --- MODEL PATH ---
# After uploading to Kaggle Dataset, use the local path below.
# First time: the dataset gets mounted at /kaggle/input/qwen3-vl-30b/
MODEL_ID = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"

# Video path (upload video to Kaggle Datasets first)
VIDEO_PATH = "/kaggle/input/datasets/yoofun/whitesummer"  # Update this!

# Whisper settings
USE_WHISPER = False              # Set False to skip transcription
WHISPER_MODEL = "medium"        # tiny, base, small, medium, large-v3
WHISPER_LANGUAGE = None          # None = auto-detect, or "en", "zh", etc.

# LLM settings (local Qwen3-VL-30B on dual T4)
USE_VLM = True                  # Set False to skip VLM descriptions
USE_STAGE2 = True               # Set False to skip Stage 2 summarization
VIDEO_TYPE = "movie"            # "movie" or "tv_series"

# ============================================

In [ ]:
# Download model & save as permanent Kaggle Dataset (run ONCE, T4x2 OFF)
import os, shutil, json, subprocess

# Clean EVERYTHING from previous failed attempts
for path in ["/kaggle/working/qwen_model", "/kaggle/tmp/qwen_model",
             "/kaggle/tmp/hf_cache", os.path.expanduser("~/.cache/huggingface")]:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)

MODEL = "unsloth/Qwen3-VL-32B-Instruct-unsloth-bnb-4bit"
DST = "/kaggle/tmp/qwen_model"
os.makedirs(DST, exist_ok=True)

# Use hf CLI (not hugggingface-cli, which is deprecated)
print(f"Downloading {MODEL} to {DST}...")
import subprocess
ret = subprocess.call(["hf", "download", MODEL, "--local-dir", DST])
if ret != 0:
    raise RuntimeError(f"Download failed (exit {ret})")

files = os.listdir(DST)
print(f"Downloaded {len(files)} files")

print("Creating Kaggle Dataset...")
with open(os.path.join(DST, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "qwen3-vl-32b-4bit",
        "id": "yoofun/qwen3-vl-32b-4bit",
        "licenses": [{"name": "Apache-2.0"}]
    }, f)

!kaggle datasets create -p {DST} --dir-mode zip
print("\nDONE! Add this dataset from the Data panel.")

In [ ]:
# Load Faster-Whisper model
whisper_model = None

if USE_WHISPER:
    from faster_whisper import WhisperModel
    
    print(f"Loading Whisper model: {WHISPER_MODEL}")
    # Use float16 on GPU, int8 on CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    whisper_model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
    print(f"Whisper loaded on {device} with {compute_type}")
else:
    print("Whisper disabled")

In [ ]:
# Load LLM via Unsloth FastVisionModel (bnb-4bit, optimized)
llm_model = None
llm_processor = None

if USE_VLM or USE_STAGE2:
    import bitsandbytes
    print(f"bitsandbytes: {bitsandbytes.__version__}")
    
    from unsloth import FastVisionModel
    
    print(f"Loading LLM (Unsloth): {MODEL_ID}")
    print(f"GPU count: {torch.cuda.device_count()}")
    
    llm_model, llm_processor = FastVisionModel.from_pretrained(
        model_name=MODEL_ID,
        load_in_4bit=True,
        low_cpu_mem_usage=True,
    )
    
    llm_model = FastVisionModel.for_inference(llm_model)
    
    print("LLM loaded via Unsloth!")
else:
    print("VLM/Stage2 disabled, skipping LLM load")

In [ ]:
# Shot detection
def detect_shots(video_path, threshold=27.0):
    scene_list = detect(video_path, AdaptiveDetector(adaptive_threshold=threshold))
    shots = []
    for idx, (start, end) in enumerate(scene_list):
        shots.append({
            "shot_id": idx + 1,
            "start_time": start.get_seconds(),
            "end_time": end.get_seconds()
        })
    return shots

shots = detect_shots(VIDEO_PATH)
print(f"Found {len(shots)} shots")
pd.DataFrame(shots)

In [ ]:
# Whisper transcription
def transcribe_video(video_path, model, language=None):
    segments, info = model.transcribe(
        video_path,
        language=language,
        beam_size=5,
        vad_filter=True
    )
    
    subtitles = []
    for seg in segments:
        subtitles.append({
            "text": seg.text.strip(),
            "start_time": seg.start,
            "end_time": seg.end
        })
    return subtitles

def find_dialogue_gaps(subtitles, shots, min_gap_duration=0.5):
    """Find time intervals without dialogue for AD placement."""
    gaps = []
    for shot in shots:
        shot_start = shot["start_time"]
        shot_end = shot["end_time"]
        
        shot_subs = [s for s in subtitles 
                     if s["start_time"] < shot_end and s["end_time"] > shot_start]
        shot_subs.sort(key=lambda x: x["start_time"])
        
        current_time = shot_start
        for sub in shot_subs:
            if sub["start_time"] > current_time + min_gap_duration:
                gaps.append({
                    "shot_id": shot["shot_id"],
                    "start_time": current_time,
                    "end_time": sub["start_time"]
                })
            current_time = max(current_time, sub["end_time"])
        
        if shot_end > current_time + min_gap_duration:
            gaps.append({
                "shot_id": shot["shot_id"],
                "start_time": current_time,
                "end_time": shot_end
            })
    return gaps

if USE_WHISPER and whisper_model:
    print("Transcribing video...")
    subtitles = transcribe_video(VIDEO_PATH, whisper_model, WHISPER_LANGUAGE)
    print(f"Found {len(subtitles)} subtitle segments")
    
    dialogue_gaps = find_dialogue_gaps(subtitles, shots)
    print(f"Found {len(dialogue_gaps)} dialogue gaps for AD placement")
else:
    print("Whisper disabled")
    subtitles = []
    dialogue_gaps = []

In [ ]:
# Merge shots and subtitles
def merge_shots_subtitles(shots, subtitles):
    results = []
    for shot in shots:
        overlapping = [s for s in subtitles 
                      if s["start_time"] < shot["end_time"] and s["end_time"] > shot["start_time"]]
        subtitle_text = " ".join([s["text"] for s in overlapping]) if overlapping else ""
        results.append({
            "shot_id": shot["shot_id"],
            "start_time": shot["start_time"],
            "end_time": shot["end_time"],
            "subtitle": subtitle_text
        })
    return pd.DataFrame(results)

merged_df = merge_shots_subtitles(shots, subtitles)
merged_df

In [ ]:
# VLM description (Stage 1) using transformers
import warnings
warnings.filterwarnings("ignore", message=".*Both max_new_tokens and max_length.*")

def extract_frames(video_path, shot, num_frames=8):
    """Extract frames from video shot as PIL Images."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_frame = int(shot["start_time"] * fps)
    end_frame = int(shot["end_time"] * fps)
    frame_indices = np.linspace(start_frame, end_frame, num_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
    cap.release()
    return frames

def describe_frames_local(frames, model, processor):
    """Get VLM description using Qwen3-VL/Qwen2.5-VL via Unsloth."""
    prompt = "請簡短描述這段影片片段發生了什麼事。請使用繁體中文。專注於角色、動作和環境。"
    
    content = [{"type": "text", "text": prompt}]
    for frame in frames:
        content.append({"type": "image", "image": frame})
    messages = [{"role": "user", "content": content}]
    
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)
    
    generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)]
    response = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return response

def summarize_to_ad_local(desc, model, processor, word_limit=15):
    """Summarize description to concise AD sentence using Qwen3-VL."""
    prompt = f"""請將以下描述濃縮成一句簡潔的口述影像句子。使用繁體中文。
專注於角色、動作和關鍵物件。
使用名字或代名詞。避免提到鏡頭。
限制在 {word_limit} 個字以內。

Input: {desc}

Output:"""
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=128)
    
    generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)]
    result = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()
    if not result.endswith("."):
        result += "."
    return result

# Run VLM if enabled
descriptions_dict = {}
if USE_VLM and llm_model:
    print("Running Stage 1: VLM descriptions...")
    for shot in shots:
        try:
            frames = extract_frames(VIDEO_PATH, shot)
            desc = describe_frames_local(frames, llm_model, llm_processor)
            descriptions_dict[shot["shot_id"]] = desc
            print(f"Shot {shot['shot_id']}: OK")
        except Exception as e:
            print(f"Shot {shot['shot_id']}: Failed - {e}")
            descriptions_dict[shot["shot_id"]] = ""
    
    merged_df["video_description"] = merged_df["shot_id"].map(descriptions_dict).fillna("")
    
    stage1_df = merged_df.copy()
    stage1_df.to_csv("/kaggle/working/stage1_descriptions.csv", index=False)
    print("Stage 1 saved to: stage1_descriptions.csv")
else:
    print("VLM disabled")

# Run Stage 2 if enabled
if USE_STAGE2 and llm_model and descriptions_dict:
    print("\nRunning Stage 2: Summarizing to AD...")
    ad_sentences = []
    for _, row in merged_df.iterrows():
        duration = row["end_time"] - row["start_time"]
        word_limit = max(1, int(duration * 3))  # 3 words per second
        try:
            ad = summarize_to_ad_local(row["video_description"], llm_model, llm_processor, word_limit)
            ad_sentences.append(ad)
        except:
            ad_sentences.append("")
    
    merged_df["ad_sentence"] = ad_sentences
    
    stage2_df = merged_df[["shot_id", "start_time", "end_time", "ad_sentence"]].copy()
    stage2_df.to_csv("/kaggle/working/stage2_audio_descriptions.csv", index=False)
    print("Stage 2 saved to: stage2_audio_descriptions.csv")
else:
    print("Stage 2 disabled")

merged_df

In [ ]:
# Save outputs
output_path = "/kaggle/working/shot_by_shot_output.csv"
merged_df.to_csv(output_path, index=False)
print(f"Main output saved to: {output_path}")

print("\nOutput files:")
print("  - shot_by_shot_output.csv")
if USE_VLM and llm_model:
    print("  - stage1_descriptions.csv")
if USE_STAGE2 and llm_model:
    print("  - stage2_audio_descriptions.csv")

In [ ]:
# Download links
from IPython.display import FileLink, display

print("Download links:")
display(FileLink("/kaggle/working/shot_by_shot_output.csv"))
if USE_VLM and llm_model:
    display(FileLink("/kaggle/working/stage1_descriptions.csv"))
if USE_STAGE2 and llm_model:
    display(FileLink("/kaggle/working/stage2_audio_descriptions.csv"))